In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

class DampedOscillator(nn.Module):
    def __init__(self, dt):
        super().__init__()
        self.m = nn.Parameter(torch.tensor(1.0))
        self.l = nn.Parameter(torch.tensor(1.0))
        self.b = nn.Parameter(torch.tensor(1.0))

        self.g = 9.81
        self.dt = dt
    
    def forward(self, theta0, ts):
        theta0 = torch.tensor(theta0, dtype=torch.float32)

        beta = self.b/(2*self.m)
        omega0 = torch.sqrt(self.g / self.l)

        omegas, thetas = [omega0], [theta0]

        for _ in range(ts - 1):
            theta = thetas[-1]
            omega = omegas[-1]
            
            omega_new = omega + self.dt * (-2 * beta * omega - torch.sin(theta) * omega0**2)
            
            theta_new = theta + self.dt * omega_new
            
            thetas.append(theta_new)
            omegas.append(omega_new)

        return torch.stack(thetas)

def generate_data(T, dt, m = 1, l = 2.25, b = 0.35, g = 9.81):
    Ts = np.arange(0, T + dt, dt)
    beta = b / (2 * m)

    theta0 = 120 * np.pi / 180
    omega0 = np.sqrt(g / l)

    omegas, thetas = [omega0], [theta0]

    for _ in range(len(Ts)+1):
        theta = thetas[-1]
        omega = omegas[-1]
        
        omega_new = omega + dt * (-2 * beta * omega - np.sin(theta) * omega0**2)
        
        theta_new = theta + dt * omega_new
        
        thetas.append(theta_new)
        omegas.append(omega_new)

    return np.stack(thetas)


T, dt = 10, 0.005
m, l, b = 1.0, 1.5, 0.5

true_trajectory = torch.tensor(generate_data(T, dt, m, l, b), dtype=torch.float32)

# --- TRAIN --- #
model = DampedOscillator(dt)
optim = torch.optim.Adam(model.parameters(), 1e-2)

criterion = nn.MSELoss()
epochs = 100

losses = []
for epoch in range(epochs):
    optim.zero_grad()

    theta0 = true_trajectory[0]
    pred_traj = model(theta0, len(true_trajectory))

    loss = criterion(pred_traj, true_trajectory)

    loss.backward()
    optim.step()

    losses.append(loss.item())

    if epoch % 10 == 0:
        print(f'Epoch {epoch}/{epochs}, Loss: {loss.item():.6f}')
        # print(f'Current omega: {model.omega.item():.3f}, Current a: {model.a.item():.3f}')

# Plot results
plt.figure(figsize=(12, 4))

plt.subplot(121)
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')

plt.subplot(122)
plt.plot(true_trajectory.detach().numpy(), label='True')
plt.plot(pred_traj.detach().numpy(), '--', label='Predicted')
plt.xlabel('Time')
plt.ylabel('Position')
plt.title('Trajectories')
plt.legend()

plt.tight_layout()
plt.show()


print("\nFinal Parameters:")
print(f"Learned m: {model.m.item():.3f}, Ground Truth: {m}")
print(f"Learned l: {model.l.item():.3f}, Ground Truth: {l}")
print(f"Learned b: {model.b.item():.3f}, Ground Truth: {b}")